In [3]:
import os
import json
import random
import logging
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import Qwen2Tokenizer, Qwen2Model
from peft import get_peft_model, LoraConfig, TaskType

# ================== 1. 字体环境安全处理 ==================
import matplotlib
matplotlib.use('Agg')  # 必须在 import pyplot 之前设置，避免 display 问题

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 清理所有可能残留的自定义字体引用
fm.fontManager.ttflist = [f for f in fm.fontManager.ttflist 
                          if 'custom_font' not in f.fname.lower()]

# 使用系统内置的 CJK 字体（AutoDL 环境通常有这几个）
CJK_CANDIDATES = [
    'WenQuanYi Micro Hei',
    'WenQuanYi Zen Hei', 
    'Noto Sans CJK SC',
    'SimHei',
    'DejaVu Sans',  # 最后兜底，不支持中文但不报错
]

selected_font = 'DejaVu Sans'
available = {f.name for f in fm.fontManager.ttflist}
for name in CJK_CANDIDATES:
    if name in available:
        selected_font = name
        break

plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False
print(f"ℹ️ 使用字体: {selected_font}")

# ================== 1. 基础配置 ==================
os.environ['TRANSFORMERS_OFFLINE'] = '1'
MODEL_PATH = '/root/autodl-tmp/gte-Qwen2-1.5B-instruct'
SAVE_DIR = '/root/autodl-tmp/foodcf_encoder_output'
TRAIN_FILE = '/root/autodl-tmp/encoder_train.jsonl'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 核心常量定义
EPOCHS = 10
BATCH_SIZE = 8       # 4090D 三元组训练的安全值
LR = 1e-4           # 适配小 Batch 的学习率
LORA_RANK = 64      # LoRA 秩
EMBED_DIM = 128     # 最终映射的向量维度
MAX_LENGTH = 192    # 文本截断长度

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('foodcf_encoder')
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------- 模型组件 ----------
class LatentAttnModule(nn.Module):
    def __init__(self, hidden_dim, num_tokens=4, output_dim=128):
        super().__init__()
        self.latent_queries = nn.Parameter(torch.randn(1, num_tokens, hidden_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.proj = nn.Linear(num_tokens * hidden_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)

    def forward(self, hidden_states):
        batch_size = hidden_states.size(0)
        queries = self.latent_queries.expand(batch_size, -1, -1).to(hidden_states.dtype)
        attn_out, _ = self.cross_attn(queries, hidden_states, hidden_states)
        flat = attn_out.reshape(batch_size, -1)
        return self.norm(self.proj(flat))

# ---------- 损失函数 ----------
def info_nce_loss(anchor, positive, negatives, temperature=0.07):
    logits_matrix = torch.mm(anchor, positive.t()) / temperature
    labels = torch.arange(anchor.size(0), device=anchor.device)
    loss_a2p = F.cross_entropy(logits_matrix, labels)
    loss_p2a = F.cross_entropy(logits_matrix.t(), labels)
    return (loss_a2p + loss_p2a) / 2

class TripletDataset(Dataset):
    def __init__(self, records, tokenizer):
        self.records = records
        self.tokenizer = tokenizer
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        # 合并 Tokenization 逻辑
        t_params = {"max_length": MAX_LENGTH, "truncation": True, "padding": 'max_length', "return_tensors": 'pt'}
        q = self.tokenizer("判断该用户会在哪些外卖餐厅下单：" + r['query'], **t_params)
        p = self.tokenizer("描述这家外卖餐厅的特色：" + r['positive'], **t_params)
        n = self.tokenizer("描述这家外卖餐厅的特色：" + r['hard_negative'], **t_params)
        return {'q_ids': q['input_ids'].squeeze(0), 'q_mask': q['attention_mask'].squeeze(0),
                'p_ids': p['input_ids'].squeeze(0), 'p_mask': p['attention_mask'].squeeze(0),
                'n_ids': n['input_ids'].squeeze(0), 'n_mask': n['attention_mask'].squeeze(0)}

# ================== 2. 初始化流程 ==================
records = [json.loads(line) for line in open(TRAIN_FILE, 'r', encoding='utf-8') if line.strip()]
logger.info(f'✅ 加载 {len(records)} 条样本')

tokenizer = Qwen2Tokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
base_model = Qwen2Model.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.bfloat16).to(DEVICE)
base_model.gradient_checkpointing_enable() # 开启梯度检查点

# 配置 LoRA
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION, r=LORA_RANK, lora_alpha=LORA_RANK*2,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'], lora_dropout=0.05
)
base_model = get_peft_model(base_model, lora_config)

# 初始化 Pooling (确保 bf16)
user_pool = LatentAttnModule(base_model.config.hidden_size, output_dim=EMBED_DIM).to(DEVICE).to(torch.bfloat16)
rest_pool = LatentAttnModule(base_model.config.hidden_size, output_dim=EMBED_DIM).to(DEVICE).to(torch.bfloat16)

# 数据分配
random.shuffle(records)
split = int(len(records) * 0.9)
train_loader = DataLoader(TripletDataset(records[:split], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TripletDataset(records[split:], tokenizer), batch_size=BATCH_SIZE)

optimizer = optim.AdamW([
    {'params': user_pool.parameters(), 'lr': LR},
    {'params': rest_pool.parameters(), 'lr': LR},
    {'params': [p for p in base_model.parameters() if p.requires_grad], 'lr': LR * 0.1}
])

def encode_batch(ids, mask, pool):
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        outputs = base_model(input_ids=ids, attention_mask=mask)
        return pool(outputs.last_hidden_state)

# ================== 3. 训练循环 ==================
best_val_loss = float('inf')
logger.info("🚀 启动显存优化训练...")

for epoch in range(1, EPOCHS + 1):
    base_model.train(); user_pool.train(); rest_pool.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        q_emb = encode_batch(batch['q_ids'].to(DEVICE), batch['q_mask'].to(DEVICE), user_pool)
        p_emb = encode_batch(batch['p_ids'].to(DEVICE), batch['p_mask'].to(DEVICE), rest_pool)
        n_emb = encode_batch(batch['n_ids'].to(DEVICE), batch['n_mask'].to(DEVICE), rest_pool)
        
        loss = info_nce_loss(F.normalize(q_emb, dim=-1), F.normalize(p_emb, dim=-1), F.normalize(n_emb, dim=-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # 验证逻辑
    base_model.eval(); val_loss = 0.0
    with torch.no_grad():
        for b in val_loader:
            qe = encode_batch(b['q_ids'].to(DEVICE), b['q_mask'].to(DEVICE), user_pool)
            pe = encode_batch(b['p_ids'].to(DEVICE), b['p_mask'].to(DEVICE), rest_pool)
            ne = encode_batch(b['n_ids'].to(DEVICE), b['n_mask'].to(DEVICE), rest_pool)
            val_loss += info_nce_loss(F.normalize(qe, dim=-1), F.normalize(pe, dim=-1), F.normalize(ne, dim=-1)).item()
    
    avg_val = val_loss / len(val_loader)
    logger.info(f'Epoch {epoch}: Train Loss={total_loss/len(train_loader):.4f}, Val Loss={avg_val:.4f}')
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        base_model.save_pretrained(SAVE_DIR)
        torch.save(user_pool.state_dict(), os.path.join(SAVE_DIR, 'user_pooling.pt'))
        torch.save(rest_pool.state_dict(), os.path.join(SAVE_DIR, 'restaurant_pooling.pt'))
        tokenizer.save_pretrained(SAVE_DIR)
        logger.info('⭐ 保存最优模型')

# 打包
shutil.make_archive('/root/autodl-tmp/foodcf_encoder_output', 'zip', SAVE_DIR)
print(f'\n✅ 训练完成！')

2026-03-12 15:52:46,263 [INFO] ✅ 加载 5000 条样本


ℹ️ 使用字体: DejaVu Sans


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-12 15:52:50,614 [INFO] 🚀 启动显存优化训练...
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


KeyboardInterrupt: 

In [10]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ================== 必须最先执行，放在所有 plt import 之前 ==================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from sklearn.manifold import TSNE
from transformers import Qwen2Tokenizer, Qwen2Model
from peft import PeftModel

# ================== 0. 基础组件定义 ==================
class LatentAttnModule(nn.Module):
    def __init__(self, hidden_dim, num_tokens=4, output_dim=128):
        super().__init__()
        self.latent_queries = nn.Parameter(torch.randn(1, num_tokens, hidden_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.proj = nn.Linear(num_tokens * hidden_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)

    def forward(self, hidden_states):
        batch_size = hidden_states.size(0)
        queries = self.latent_queries.expand(batch_size, -1, -1).to(hidden_states.dtype)
        attn_out, _ = self.cross_attn(queries, hidden_states, hidden_states)
        flat = attn_out.reshape(batch_size, -1)
        return self.norm(self.proj(flat))

# ================== 1. 字体配置（唯一一处，彻底替换旧逻辑）==================
# 清理所有残留的自定义字体缓存
fm.fontManager.ttflist = [f for f in fm.fontManager.ttflist
                          if 'custom_font' not in f.fname.lower()]
# 重建字体缓存
fm._load_fontmanager(try_read_cache=False)

def _find_cjk_font():
    """扫描系统字体目录，找到第一个可用的中文字体文件并注册"""
    search_dirs = [
        '/usr/share/fonts',
        '/usr/local/share/fonts',
        '/root/.fonts',
        os.path.expanduser('~/.fonts'),
    ]
    cjk_keywords = ['wqy', 'wenquanyi', 'noto', 'cjk', 'chinese', 'simhei', 'simsun', 'fang']
    
    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        for root, _, files in os.walk(d):
            for fname in files:
                if not fname.lower().endswith(('.ttf', '.otf', '.ttc')):
                    continue
                if any(k in fname.lower() for k in cjk_keywords):
                    full_path = os.path.join(root, fname)
                    try:
                        # 直接注册字体文件
                        fm.fontManager.addfont(full_path)
                        prop = fm.FontProperties(fname=full_path)
                        font_name = prop.get_name()
                        print(f"✅ 找到中文字体: {font_name}  ({full_path})")
                        return font_name
                    except Exception as e:
                        print(f"⚠️ 字体注册失败 {full_path}: {e}")
    return None

cjk_font = _find_cjk_font()

if cjk_font:
    plt.rcParams['font.family'] = cjk_font
    print(f"✅ 已设置字体: {cjk_font}")
else:
    # 最后兜底：尝试已知字体名
    fallback_names = ['WenQuanYi Micro Hei', 'WenQuanYi Zen Hei',
                      'Noto Sans CJK SC', 'SimHei']
    available = {f.name for f in fm.fontManager.ttflist}
    for name in fallback_names:
        if name in available:
            plt.rcParams['font.family'] = name
            print(f"✅ 兜底字体: {name}")
            break
    else:
        print("⚠️ 未找到中文字体，将显示方块。运行: apt-get install -y fonts-wqy-microhei 后重试")

plt.rcParams['axes.unicode_minus'] = False

# ================== 2. 模型加载 ==================
CHECKPOINT_PATH = '/root/autodl-tmp/foodcf_encoder_output'
BASE_MODEL_PATH  = '/root/autodl-tmp/gte-Qwen2-1.5B-instruct'

print("🔄 正在加载微调模型权重...")
tokenizer = Qwen2Tokenizer.from_pretrained(CHECKPOINT_PATH)
base  = Qwen2Model.from_pretrained(BASE_MODEL_PATH, torch_dtype=torch.bfloat16).cuda()
model = PeftModel.from_pretrained(base, CHECKPOINT_PATH).cuda()
model.eval()

u_pool = LatentAttnModule(base.config.hidden_size, output_dim=128).cuda().to(torch.bfloat16)
r_pool = LatentAttnModule(base.config.hidden_size, output_dim=128).cuda().to(torch.bfloat16)
u_pool.load_state_dict(torch.load(f"{CHECKPOINT_PATH}/user_pooling.pt"))
r_pool.load_state_dict(torch.load(f"{CHECKPOINT_PATH}/restaurant_pooling.pt"))

def get_vector(text, mode="user"):
    prefix = "判断该用户会在哪些外卖餐厅下单：" if mode == "user" else "描述这家外卖餐厅的特色："
    inputs = tokenizer(prefix + text, return_tensors="pt", padding=True,
                       truncation=True, max_length=192).to("cuda")
    with torch.no_grad():
        out = model(**inputs)
        vec = u_pool(out.last_hidden_state) if mode == "user" else r_pool(out.last_hidden_state)
    return F.normalize(vec, dim=-1)

# ================== 3. t-SNE 数据准备 ==================
categories = {
    "川湘麻辣": ["麻辣火锅", "剁椒鱼头", "辣子鸡", "毛血旺", "酸辣粉"],
    "清淡养生": ["冰糖炖雪梨", "清蒸鲈鱼", "养生小米粥", "山药排骨汤", "白灼生菜"],
    "西式快餐": ["芝士汉堡", "香脆薯条", "奥尔良烤翅", "夏威夷披萨", "热狗棒"],
    "精致甜点": ["提拉米苏", "杨枝甘露", "珍珠奶茶", "舒芙蕾", "舒芙蕾松饼"],
}

vectors, labels, names = [], [], []
cat_names = list(categories.keys())
for idx, cat in enumerate(cat_names):
    for item in categories[cat]:
        vec = get_vector(item, mode="rest").cpu().float().numpy()
        vectors.append(vec.flatten())
        labels.append(idx)
        names.append(item)

print("正在执行 t-SNE 降维计算...")
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embeds_2d = tsne.fit_transform(np.array(vectors))

# ================== 4. 绘图 ==================
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

for idx, cat in enumerate(cat_names):
    mask = np.array(labels) == idx
    ax.scatter(embeds_2d[mask, 0], embeds_2d[mask, 1],
               c=colors[idx], label=cat, s=200, edgecolors='white', alpha=0.8)
    for i, txt in enumerate(np.array(names)[mask]):
        ax.annotate(txt,
                    (embeds_2d[mask][i, 0], embeds_2d[mask][i, 1]),
                    fontsize=10, xytext=(5, 5), textcoords='offset points')

ax.set_title("微调模型语义空间聚类可视化", fontsize=16)
ax.legend(title="菜系标签")
ax.grid(True, linestyle=':', alpha=0.6)

out_path = '/root/autodl-tmp/semantic_clustering_result.png'
fig.savefig(out_path, bbox_inches='tight', dpi=300)
plt.close(fig)
print(f"图片已保存至: {out_path}")

# ================== 5. 语义检索 Hard Case ==================
print("\n" + "=" * 50)
print("语义检索实战演示 (Hard Case)")
print("=" * 50)

test_queries   = ["嗓子不舒服，想吃点温润清淡的，拒绝辛辣。", "想吃点重口味的肉类，越麻辣越好。"]
candidate_pool = ["老成都红油冒菜", "冰糖银耳羹", "皮蛋瘦肉粥", "麦辣鸡腿堡", "酸辣粉"]
candidate_vecs = torch.cat([get_vector(r, mode="rest") for r in candidate_pool])

for query in test_queries:
    q_vec = get_vector(query, mode="user")
    scores = torch.mm(q_vec, candidate_vecs.t())
    top_indices = torch.topk(scores, k=2).indices[0].cpu().numpy()
    print(f"\n用户需求: {query}")
    for i, idx in enumerate(top_indices):
        print(f"推荐 {i+1}: {candidate_pool[idx]} (得分: {scores[0, idx]:.4f})")

2026-03-12 16:54:32,084 [INFO] generated new fontManager


✅ 找到中文字体: WenQuanYi Micro Hei  (/usr/share/fonts/truetype/wqy/wqy-microhei.ttc)
✅ 已设置字体: WenQuanYi Micro Hei
🔄 正在加载微调模型权重...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

正在执行 t-SNE 降维计算...
图片已保存至: /root/autodl-tmp/semantic_clustering_result.png

语义检索实战演示 (Hard Case)

用户需求: 嗓子不舒服，想吃点温润清淡的，拒绝辛辣。
推荐 1: 冰糖银耳羹 (得分: 0.4551)
推荐 2: 皮蛋瘦肉粥 (得分: 0.4141)

用户需求: 想吃点重口味的肉类，越麻辣越好。
推荐 1: 麦辣鸡腿堡 (得分: 0.3398)
推荐 2: 老成都红油冒菜 (得分: 0.2676)


In [9]:
print("\n" + "消融实验：语义对齐能力对比".center(60, "="))

# 加载原生模型作为基准
raw_base = Qwen2Model.from_pretrained(BASE_MODEL_PATH, torch_dtype=torch.bfloat16).cuda()
raw_base.eval()

def get_baseline_vector(text, mode="user"):
    """
    原生模型向量提取 —— 与微调模型使用完全相同的提取方式
    唯一区别：不经过训练后的投影层
    """
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
    with torch.no_grad():
        out = raw_base(**inputs)
        # ✅ 与 get_vector 保持一致：使用 Mean Pooling
        attention_mask = inputs["attention_mask"]
        token_embeddings = out.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        vec = torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return F.normalize(vec, dim=-1)


# ── 测试样例 ──────────────────────────────────────────────
test_query = "感冒了喉咙痛，想吃点清淡温润的，不要辣。"
test_candidates = ["冰糖银耳羹", "老成都红油冒菜", "皮蛋瘦肉粥", "酸辣粉"]

print(f"\n测试需求: {test_query}\n")
print(f"{'候选':<15} | {'Base 得分':^20} | {'Ours 得分':^20} | {'提升度':^10} | 判断")
print("-" * 90)

for cand in test_candidates:
    # 原生模型：同样的 pooling 方式，无投影层
    v_base_q = get_baseline_vector(test_query, mode="user")
    v_base_r = get_baseline_vector(cand,       mode="rest")
    score_base = torch.mm(v_base_q, v_base_r.t()).item()

    # 微调模型：带投影层
    v_ft_q = get_vector(test_query, mode="user")
    v_ft_r = get_vector(cand,       mode="rest")
    score_ft = torch.mm(v_ft_q, v_ft_r.t()).item()

    improvement = (score_ft - score_base) / abs(score_base) * 100
    arrow = "↑" if improvement > 0 else "↓"

    # 语义上是否"应该推荐"
    should_recommend = cand in ["冰糖银耳羹", "皮蛋瘦肉粥"]
    expected = "应推荐" if should_recommend else "应过滤"

    print(f"{cand:<15} | {score_base:^20.4f} | {score_ft:^20.4f} | {improvement:>+7.1f}% {arrow} | {expected}")


# ── 汇总：正负样本区分度对比 ─────────────────────────────
print("\n" + "─" * 90)
print("区分度分析（正样本均值 vs 负样本均值）\n")

pos_items = ["冰糖银耳羹", "皮蛋瘦肉粥"]
neg_items = ["老成都红油冒菜", "酸辣粉"]

def avg_score(items, vec_fn_q, vec_fn_r, query):
    scores = []
    for item in items:
        q = vec_fn_q(query)
        r = vec_fn_r(item)
        scores.append(torch.mm(q, r.t()).item())
    return sum(scores) / len(scores)

base_pos = avg_score(pos_items, get_baseline_vector, get_baseline_vector, test_query)
base_neg = avg_score(neg_items, get_baseline_vector, get_baseline_vector, test_query)
ft_pos   = avg_score(pos_items, lambda t: get_vector(t, mode="user"), lambda t: get_vector(t, mode="rest"), test_query)
ft_neg   = avg_score(neg_items, lambda t: get_vector(t, mode="user"), lambda t: get_vector(t, mode="rest"), test_query)

print(f"{'':20} {'Base 模型':^20} {'微调模型 (Ours)':^20}")
print(f"{'正样本均值':20} {base_pos:^20.4f} {ft_pos:^20.4f}")
print(f"{'负样本均值':20} {base_neg:^20.4f} {ft_neg:^20.4f}")
print(f"{'正负分差 (↑越好)':20} {base_pos - base_neg:^20.4f} {ft_pos - ft_neg:^20.4f}")

gap_improvement = ((ft_pos - ft_neg) - (base_pos - base_neg)) / abs(base_pos - base_neg) * 100
print(f"\n  语义区分度提升: {gap_improvement:+.1f}%")
print(f"  {'微调有效：模型学会了区分清淡 vs 辛辣' if gap_improvement > 0 else '❌ 微调未能提升区分度，请检查训练数据或损失函数'}")


=======================消融实验：语义对齐能力对比========================


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


测试需求: 感冒了喉咙痛，想吃点清淡温润的，不要辣。

候选              |       Base 得分        |       Ours 得分        |    提升度     | 判断
------------------------------------------------------------------------------------------
冰糖银耳羹           |        0.5948        |        0.4375        |   -26.4% ↓ | 应推荐
老成都红油冒菜         |        0.5044        |        0.3164        |   -37.3% ↓ | 应过滤
皮蛋瘦肉粥           |        0.5578        |        0.4102        |   -26.5% ↓ | 应推荐
酸辣粉             |        0.4937        |        0.3574        |   -27.6% ↓ | 应过滤

──────────────────────────────────────────────────────────────────────────────────────────
区分度分析（正样本均值 vs 负样本均值）

                           Base 模型            微调模型 (Ours)     
正样本均值                       0.5763               0.4238       
负样本均值                       0.4990               0.3369       
正负分差 (↑越好)                  0.0772               0.0869       

  语义区分度提升: +12.5%
  微调有效：模型学会了区分清淡 vs 辛辣
